# Device fidelity, statistical analysis, and timing sensitivity

This notebook reproduces the main device-to-reference analyses reported in the manuscript.

**Input:** extracted experimental data under `data/` (TDMS reference files and the 56 retained device Excel recordings).

**Outputs:** quality-control tables, common run-level alignment, the 56-row fidelity table, repeated-measures/FDR results, post-hoc comparisons, and residual-timing/SNR sensitivity results under `results/`.


In [ ]:
%pip -q install nptdms openpyxl statsmodels

from pathlib import Path
from fractions import Fraction
from itertools import combinations
import re
import unicodedata
import zipfile

import numpy as np
import pandas as pd
from scipy import signal, stats
from nptdms import TdmsFile
from statsmodels.stats.anova import AnovaRM
from statsmodels.stats.multitest import multipletests

cwd = Path.cwd()
if (cwd / "data").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data").exists():
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Data directory not found: {DATA_DIR}\n"
        "Create a 'data' directory in the repository and place the experimental data there."
    )

for archive in DATA_DIR.glob("*.zip"):
    with zipfile.ZipFile(archive, "r") as zf:
        zf.extractall(DATA_DIR)

RUN_MAPPING = [
    ("bingol", 30, "111445"), ("bingol", 50, "111811"),
    ("kirkhan", 30, "112036"), ("kirkhan", 50, "112343"),
    ("sindirgi", 30, "112705"), ("sindirgi", 50, "112951"),
    ("sulusaray", 30, "113156"), ("sulusaray", 50, "113737"),
    ("bozkurt", 30, "114254"), ("bozkurt", 50, "114537"),
    ("goksun", 30, "114901"), ("goksun", 50, "115059"),
    ("van", 30, "115302"), ("van", 50, "115537"),
]

def clean_text(value):
    value = str(value).lower().translate(
        str.maketrans({"ı":"i","ö":"o","ü":"u","ş":"s","ğ":"g","ç":"c"})
    )
    value = unicodedata.normalize("NFKD", value)
    return "".join(ch for ch in value if not unicodedata.combining(ch))

def identify_event(name):
    s = clean_text(name)
    if "sulusaray" in s: return "sulusaray"
    if "bozkurt" in s: return "bozkurt"
    if "bing" in s: return "bingol"
    if "kirkhan" in s or "rikhan" in s or "rkhan" in s: return "kirkhan"
    if "sindirgi" in s or ("nd" in s and "rg" in s): return "sindirgi"
    if "goksun" in s or "ksun" in s: return "goksun"
    if "van" in s: return "van"
    return None

def identify_setting(name):
    s = clean_text(name).replace(" ", "")
    if "30hz" in s: return 30
    if "50hz" in s: return 50
    return None

def identify_device(path):
    s = clean_text(path)
    squashed = s.replace(" ", "")
    if "iphone14promax" in squashed:
        return "iPhone 14ProMax"
    if "iphone14" in squashed:
        return "iPhone 14"
    if "samsungs6lite" in squashed:
        return "Samsung S6 Lite"
    if re.search(r"(^|[/_\-\s])s26([/_\-\s]|$)", s):
        return "S26"
    return None

tdms_files = [p for p in DATA_DIR.rglob("*.tdms") if p.is_file()]
xlsx_files = [p for p in DATA_DIR.rglob("*.xlsx") if p.is_file()]

def find_tdms(run_id):
    matches = [p for p in tdms_files if run_id in p.name]
    if len(matches) != 1:
        raise RuntimeError(f"TDMS run {run_id}: expected one match, found {len(matches)}")
    return matches[0]

records = []
for path in xlsx_files:
    event = identify_event(path.name)
    setting = identify_setting(path.name)
    device = identify_device(path)
    if event and setting and device:
        records.append({
            "event": event,
            "nominal_fs": setting,
            "device": device,
            "path": path,
        })

phone_manifest = (
    pd.DataFrame(records)
    .drop_duplicates(subset=["event", "nominal_fs", "device"])
    .sort_values(["event", "nominal_fs", "device"])
    .reset_index(drop=True)
)

if len(phone_manifest) != 56:
    raise RuntimeError(f"Expected 56 device recordings, found {len(phone_manifest)}")

for _, _, run_id in RUN_MAPPING:
    find_tdms(run_id)

print(f"Found {len(tdms_files)} TDMS files and {len(phone_manifest)} retained device recordings.")


In [ ]:
def load_device(path):
    df = pd.read_excel(path, sheet_name="accelerometer_data")
    ts = pd.to_datetime(df["timestamp"], errors="coerce")
    x = pd.to_numeric(df["x"], errors="coerce")
    y = pd.to_numeric(df["y"], errors="coerce")
    z = pd.to_numeric(df["z"], errors="coerce")
    return df, ts, x, y, z

qc_rows = []
for _, row in phone_manifest.iterrows():
    df, ts, x, y, z = load_device(row["path"])
    valid_t = ts.notna()
    dt = ts[valid_t].diff().dt.total_seconds().dropna()

    qc_rows.append({
        "event": row["event"],
        "nominal_fs": row["nominal_fs"],
        "device": row["device"],
        "n_samples": len(df),
        "invalid_timestamps": int(ts.isna().sum()),
        "duplicate_timestamps": int(ts[valid_t].duplicated().sum()),
        "nonpositive_intervals": int((dt <= 0).sum()),
        "missing_xyz": int((x.isna() | y.isna() | z.isna()).sum()),
        "all_zero_samples": int(((x == 0) & (y == 0) & (z == 0)).sum()),
        "median_dt_s": float(dt.median()),
        "max_dt_s": float(dt.max()),
        "effective_fs_hz": float(1.0 / dt.median()),
    })

acquisition_qc = pd.DataFrame(qc_rows)
acquisition_qc.to_csv(RESULTS_DIR / "Acquisition_QC.csv", index=False)

def resample_signal(x, fs_old, fs_new):
    ratio = Fraction(float(fs_new / fs_old)).limit_denominator(10000)
    return signal.resample_poly(
        np.asarray(x, dtype=float),
        ratio.numerator,
        ratio.denominator,
    )

def common_alignment_for_run(event, nominal_fs, run_id):
    tdms = TdmsFile.read(find_tdms(run_id))
    ch1 = tdms["Output"]["Acceleration-1"]
    ch2 = tdms["Output"]["Acceleration-2"]
    a1 = np.asarray(ch1[:], dtype=float)
    a2 = np.asarray(ch2[:], dtype=float)
    tdms_fs = 1.0 / float(ch1.properties["wf_increment"])

    run_devices = phone_manifest[
        (phone_manifest["event"] == event)
        & (phone_manifest["nominal_fs"] == nominal_fs)
    ]
    if len(run_devices) != 4:
        raise RuntimeError(f"{event}-{nominal_fs}: expected four devices")

    horizontal = []
    fs_values = []

    for _, row in run_devices.iterrows():
        _, ts, x, y, _ = load_device(row["path"])
        valid = ts.notna() & x.notna() & y.notna()
        ts = ts[valid].reset_index(drop=True)
        x = x[valid].to_numpy(float)
        y = y[valid].to_numpy(float)
        t = (ts - ts.iloc[0]).dt.total_seconds().to_numpy()
        fs_phone = 1.0 / np.median(np.diff(t))
        fs_values.append(fs_phone)

        x = x - np.mean(x)
        y = y - np.mean(y)
        h = np.sqrt(x**2 + y**2)
        horizontal.append(h - np.mean(h))

    phone_fs = float(np.median(fs_values))
    min_len = min(len(x) for x in horizontal)
    consensus = np.median(
        np.vstack([x[:min_len] for x in horizontal]),
        axis=0,
    )

    tdms_h = np.sqrt(a1**2 + a2**2)
    tdms_h = tdms_h - np.mean(tdms_h)
    reference = resample_signal(tdms_h, tdms_fs, phone_fs)

    if len(reference) > len(consensus):
        raise RuntimeError(f"{event}-{nominal_fs}: reference longer than device recording")

    correlations = np.full(len(consensus) - len(reference) + 1, np.nan)
    ref_std = np.std(reference)

    for start in range(len(correlations)):
        segment = consensus[start:start + len(reference)]
        if ref_std > 0 and np.std(segment) > 0:
            correlations[start] = np.corrcoef(reference, segment)[0, 1]

    best_start = int(np.nanargmax(correlations))
    return {
        "event": event,
        "nominal_fs": nominal_fs,
        "phone_fs": phone_fs,
        "tdms_fs": tdms_fs,
        "common_lag_samples": best_start,
        "common_lag_seconds": best_start / phone_fs,
        "consensus_corr": float(correlations[best_start]),
    }

common_alignment = pd.DataFrame([
    common_alignment_for_run(event, setting, run_id)
    for event, setting, run_id in RUN_MAPPING
]).sort_values(["event", "nominal_fs"]).reset_index(drop=True)

common_alignment.to_csv(RESULTS_DIR / "Common_Run_Alignment.csv", index=False)

display(acquisition_qc.groupby("nominal_fs")[["median_dt_s", "effective_fs_hz"]].median().round(4))
display(common_alignment.round(4))


In [ ]:
G0 = 9.80665
FMIN, FMAX = 0.5, 12.0
BANDS = [(0.5, 3.0), (3.0, 6.0), (6.0, 9.0), (9.0, 12.0)]

def horizontal_metrics(ref, obs):
    n = min(len(ref), len(obs))
    ref = np.asarray(ref[:n], dtype=float)
    obs = np.asarray(obs[:n], dtype=float)

    pga_ref, pga_obs = np.max(ref), np.max(obs)
    rms_ref = np.sqrt(np.mean(ref**2))
    rms_obs = np.sqrt(np.mean(obs**2))

    ref_c = ref - np.mean(ref)
    obs_c = obs - np.mean(obs)
    corr = np.corrcoef(ref_c, obs_c)[0, 1] if np.std(ref_c) > 0 and np.std(obs_c) > 0 else np.nan
    rmse = np.sqrt(np.mean((obs_c - ref_c)**2))
    ref_range = np.ptp(ref_c)
    nrmse = rmse / ref_range if ref_range > 0 else np.nan

    pga_ratio = pga_obs / pga_ref if pga_ref > 0 else np.nan
    rms_ratio = rms_obs / rms_ref if rms_ref > 0 else np.nan

    return {
        "h_corr": corr,
        "h_rmse_g": rmse,
        "h_nrmse": nrmse,
        "tdms_h_pga_g": pga_ref,
        "phone_h_pga_g": pga_obs,
        "h_pga_ratio": pga_ratio,
        "tdms_h_rms_g": rms_ref,
        "phone_h_rms_g": rms_obs,
        "h_rms_ratio": rms_ratio,
        "pga_log_error": abs(np.log(pga_ratio)) if pga_ratio > 0 else np.nan,
        "rms_log_error": abs(np.log(rms_ratio)) if rms_ratio > 0 else np.nan,
    }

def component_psd(x, fs):
    x = np.asarray(x, dtype=float) - np.mean(x)
    nperseg = min(len(x), int(fs * 8))
    return signal.welch(
        x,
        fs=fs,
        nperseg=nperseg,
        detrend="constant",
        scaling="density",
    )

def horizontal_vector_psd(x, y, fs):
    fx, px = component_psd(x, fs)
    fy, py = component_psd(y, fs)
    if not np.allclose(fx, fy):
        raise RuntimeError("PSD frequency grids do not match")
    return fx, px + py

def spectral_metrics(f_ref, p_ref, f_obs, p_obs):
    grid = np.linspace(FMIN, FMAX, 500)
    pr = np.interp(grid, f_ref, p_ref)
    po = np.interp(grid, f_obs, p_obs)
    eps = np.finfo(float).eps

    psd_corr = np.corrcoef(np.log10(pr + eps), np.log10(po + eps))[0, 1]
    dom_ref = grid[np.argmax(pr)]
    dom_obs = grid[np.argmax(po)]
    cent_ref = np.sum(grid * pr) / np.sum(pr)
    cent_obs = np.sum(grid * po) / np.sum(po)

    er, eo = [], []
    for lo, hi in BANDS:
        mask = (grid >= lo) & (grid < hi)
        er.append(np.trapezoid(pr[mask], grid[mask]))
        eo.append(np.trapezoid(po[mask], grid[mask]))

    er = np.asarray(er, float)
    eo = np.asarray(eo, float)
    er /= er.sum()
    eo /= eo.sum()

    return {
        "psd_corr": psd_corr,
        "dom_freq_error_hz": abs(dom_obs - dom_ref),
        "centroid_error_hz": abs(cent_obs - cent_ref),
        "band_energy_distance": np.sum(np.abs(er - eo)),
    }

rows = []

for _, align in common_alignment.iterrows():
    event = align["event"]
    nominal_fs = int(align["nominal_fs"])
    lag = int(align["common_lag_samples"])
    run_id = next(r for e, s, r in RUN_MAPPING if e == event and s == nominal_fs)

    tdms = TdmsFile.read(find_tdms(run_id))
    ch1 = tdms["Output"]["Acceleration-1"]
    ch2 = tdms["Output"]["Acceleration-2"]
    tx = np.asarray(ch1[:], dtype=float)
    ty = np.asarray(ch2[:], dtype=float)
    tdms_fs = 1.0 / float(ch1.properties["wf_increment"])
    tdms_h = np.sqrt(tx**2 + ty**2)

    run_devices = phone_manifest[
        (phone_manifest["event"] == event)
        & (phone_manifest["nominal_fs"] == nominal_fs)
    ]

    for _, row in run_devices.iterrows():
        _, ts, x, y, _ = load_device(row["path"])
        valid = ts.notna() & x.notna() & y.notna()
        ts = ts[valid].reset_index(drop=True)
        x = x[valid].to_numpy(float)
        y = y[valid].to_numpy(float)

        t = (ts - ts.iloc[0]).dt.total_seconds().to_numpy()
        phone_fs = 1.0 / np.median(np.diff(t))
        xg, yg = x / G0, y / G0

        x_dyn = xg - np.mean(xg)
        y_dyn = yg - np.mean(yg)
        phone_h = np.sqrt(x_dyn**2 + y_dyn**2)

        ref_h = resample_signal(tdms_h, tdms_fs, phone_fs)
        stop = lag + len(ref_h)
        if stop > len(phone_h):
            raise RuntimeError(f"Alignment exceeds recording: {event}-{nominal_fs}-{row['device']}")
        tm = horizontal_metrics(ref_h, phone_h[lag:stop])

        tx_rs = resample_signal(tx, tdms_fs, phone_fs)
        ty_rs = resample_signal(ty, tdms_fs, phone_fs)
        stop2 = lag + len(tx_rs)
        if stop2 > len(xg):
            raise RuntimeError(f"PSD alignment exceeds recording: {event}-{nominal_fs}-{row['device']}")

        px = xg[lag:stop2]
        py = yg[lag:stop2]
        n = min(len(tx_rs), len(px))

        f_ref, p_ref = horizontal_vector_psd(tx_rs[:n], ty_rs[:n], phone_fs)
        f_obs, p_obs = horizontal_vector_psd(px[:n], py[:n], phone_fs)
        fm = spectral_metrics(f_ref, p_ref, f_obs, p_obs)

        rows.append({
            "event": event,
            "nominal_fs": nominal_fs,
            "device": row["device"],
            "phone_fs": phone_fs,
            **tm,
            **fm,
        })

master_quality = (
    pd.DataFrame(rows)
    .sort_values(["event", "nominal_fs", "device"])
    .reset_index(drop=True)
)

if len(master_quality) != 56:
    raise RuntimeError(f"Expected 56 metric rows, found {len(master_quality)}")

master_quality.to_csv(RESULTS_DIR / "master_quality_56.csv", index=False)

display(
    master_quality.groupby("device")[
        ["h_corr", "h_nrmse", "psd_corr", "band_energy_distance"]
    ].mean().round(4)
)


In [ ]:
METRICS = {
    "h_corr": "Horizontal-resultant correlation",
    "h_nrmse": "Horizontal NRMSE",
    "pga_log_error": "PGA amplitude error",
    "rms_log_error": "RMS amplitude error",
    "psd_corr": "Vector-PSD correlation",
    "dom_freq_error_hz": "Dominant-frequency error",
    "centroid_error_hz": "Spectral-centroid error",
    "band_energy_distance": "Band-energy distance",
}

df = master_quality.copy()
df["event"] = df["event"].astype(str)
df["device"] = df["device"].astype(str)
df["nominal_fs"] = df["nominal_fs"].astype(str)

rm_rows = []
for metric, label in METRICS.items():
    fit = AnovaRM(
        df[["event", "device", "nominal_fs", metric]].dropna(),
        depvar=metric,
        subject="event",
        within=["device", "nominal_fs"],
    ).fit()

    table = fit.anova_table.reset_index()
    for _, row in table.iterrows():
        raw = str(row.iloc[0])
        effect = {
            "device": "Device",
            "nominal_fs": "Setting",
            "device:nominal_fs": "Device × Setting",
        }.get(raw, raw)

        F = float(row["F Value"])
        df1 = float(row["Num DF"])
        df2 = float(row["Den DF"])
        p = float(row["Pr > F"])
        eta = (F * df1) / (F * df1 + df2)

        rm_rows.append({
            "metric": label,
            "effect": effect,
            "F": F,
            "df1": df1,
            "df2": df2,
            "p": p,
            "partial_eta2": eta,
        })

rm = pd.DataFrame(rm_rows)
rm["q_FDR"] = np.nan
rm["FDR_significant"] = False

for effect in ["Device", "Setting", "Device × Setting"]:
    idx = rm["effect"].eq(effect)
    reject, q, _, _ = multipletests(rm.loc[idx, "p"], alpha=0.05, method="fdr_bh")
    rm.loc[idx, "q_FDR"] = q
    rm.loc[idx, "FDR_significant"] = reject

setting_rows = []
for metric, label in METRICS.items():
    wide = df.groupby(["event", "nominal_fs"])[metric].mean().unstack("nominal_fs")
    x30 = wide["30"].to_numpy(float)
    x50 = wide["50"].to_numpy(float)
    delta = x50 - x30
    t, p = stats.ttest_rel(x50, x30)
    sd = np.std(delta, ddof=1)
    se = sd / np.sqrt(len(delta))
    ci = stats.t.interval(0.95, df=len(delta)-1, loc=np.mean(delta), scale=se)

    setting_rows.append({
        "metric": label,
        "mean_30": np.mean(x30),
        "mean_50": np.mean(x50),
        "mean_delta_50_minus_30": np.mean(delta),
        "CI95_low": ci[0],
        "CI95_high": ci[1],
        "t": t,
        "p": p,
        "cohen_dz": np.mean(delta) / sd if sd > 0 else np.nan,
        "n_events": 7,
    })

setting = pd.DataFrame(setting_rows)

devices = list(df["device"].unique())
pair_rows = []

for metric, label in METRICS.items():
    wide = df.groupby(["event", "device"])[metric].mean().unstack("device")
    temp = []

    for a, b in combinations(devices, 2):
        x = wide[a].to_numpy(float)
        y = wide[b].to_numpy(float)
        diff = x - y
        t, p = stats.ttest_rel(x, y)
        sd = np.std(diff, ddof=1)

        temp.append({
            "metric": label,
            "device_1": a,
            "device_2": b,
            "mean_difference": np.mean(diff),
            "t": t,
            "p_raw": p,
            "cohen_dz": np.mean(diff) / sd if sd > 0 else np.nan,
            "n_events": 7,
        })

    reject, adjusted, _, _ = multipletests(
        [r["p_raw"] for r in temp], alpha=0.05, method="holm"
    )
    for row, p_adj, sig in zip(temp, adjusted, reject):
        row["p_Holm"] = p_adj
        row["significant_Holm"] = bool(sig)
        pair_rows.append(row)

pairwise = pd.DataFrame(pair_rows)

within_rows = []
for metric, label in METRICS.items():
    temp = []

    for device in devices:
        wide = df[df["device"] == device].pivot(
            index="event", columns="nominal_fs", values=metric
        )
        x30 = wide["30"].to_numpy(float)
        x50 = wide["50"].to_numpy(float)
        diff = x50 - x30
        t, p = stats.ttest_rel(x50, x30)
        sd = np.std(diff, ddof=1)

        temp.append({
            "metric": label,
            "device": device,
            "mean_30": np.mean(x30),
            "mean_50": np.mean(x50),
            "mean_delta_50_minus_30": np.mean(diff),
            "t": t,
            "p_raw": p,
            "cohen_dz": np.mean(diff) / sd if sd > 0 else np.nan,
            "n_events": 7,
        })

    reject, adjusted, _, _ = multipletests(
        [r["p_raw"] for r in temp], alpha=0.05, method="holm"
    )
    for row, p_adj, sig in zip(temp, adjusted, reject):
        row["p_Holm"] = p_adj
        row["significant_Holm"] = bool(sig)
        within_rows.append(row)

within = pd.DataFrame(within_rows)

rm.to_csv(RESULTS_DIR / "RepeatedMeasures_ANOVA_FDR.csv", index=False)
setting.to_csv(RESULTS_DIR / "SettingEffects_EventPaired.csv", index=False)
pairwise.to_csv(RESULTS_DIR / "DevicePairwise_EventLevel_Holm.csv", index=False)
within.to_csv(RESULTS_DIR / "WithinDevice_30vs50_Holm.csv", index=False)

display(rm.round(5))
display(pairwise[pairwise["significant_Holm"]].round(5))
display(within[within["significant_Holm"]].round(5))


In [ ]:
MAX_RESIDUAL_SHIFT_S = 5.0
NOISE_WINDOW_S = 5.0

def fidelity_metrics(ref, obs):
    n = min(len(ref), len(obs))
    ref = np.asarray(ref[:n], dtype=float)
    obs = np.asarray(obs[:n], dtype=float)
    ref_c = ref - np.mean(ref)
    obs_c = obs - np.mean(obs)

    corr = np.corrcoef(ref_c, obs_c)[0, 1] if np.std(ref_c) > 0 and np.std(obs_c) > 0 else np.nan
    rmse = np.sqrt(np.mean((obs_c - ref_c)**2))
    ref_range = np.ptp(ref_c)
    nrmse = rmse / ref_range if ref_range > 0 else np.nan
    return corr, nrmse

def best_pearson_start(phone_signal, ref_signal, common_start, fs, max_shift_s=5.0):
    phone_signal = np.asarray(phone_signal, dtype=float)
    ref_signal = np.asarray(ref_signal, dtype=float)
    N = len(ref_signal)
    max_shift = int(round(max_shift_s * fs))

    lo = max(0, int(common_start) - max_shift)
    hi = min(len(phone_signal) - N, int(common_start) + max_shift)
    if hi < lo:
        raise RuntimeError("No valid residual-lag search interval")

    ref_c = ref_signal - np.mean(ref_signal)
    ref_ss = np.sum(ref_c**2)

    dots = signal.correlate(phone_signal, ref_c, mode="valid", method="fft")
    cs = np.concatenate([[0.0], np.cumsum(phone_signal)])
    cs2 = np.concatenate([[0.0], np.cumsum(phone_signal**2)])

    starts = np.arange(lo, hi + 1)
    sums = cs[starts + N] - cs[starts]
    sums2 = cs2[starts + N] - cs2[starts]
    win_ss = sums2 - (sums**2) / N
    denom = np.sqrt(ref_ss * np.maximum(win_ss, 0))

    corr = np.full(len(starts), np.nan)
    valid = denom > 0
    corr[valid] = dots[starts][valid] / denom[valid]

    idx = int(np.nanargmax(corr))
    return int(starts[idx]), float(corr[idx])

def dynamic_vector_rms(x, y):
    x = np.asarray(x, dtype=float) - np.mean(x)
    y = np.asarray(y, dtype=float) - np.mean(y)
    return np.sqrt(np.mean(x**2 + y**2))

diagnostic_rows = []

for _, align in common_alignment.iterrows():
    event = align["event"]
    nominal_fs = int(align["nominal_fs"])
    common_lag = int(align["common_lag_samples"])
    run_id = next(r for e, s, r in RUN_MAPPING if e == event and s == nominal_fs)

    tdms = TdmsFile.read(find_tdms(run_id))
    ch1 = tdms["Output"]["Acceleration-1"]
    ch2 = tdms["Output"]["Acceleration-2"]
    tx = np.asarray(ch1[:], dtype=float)
    ty = np.asarray(ch2[:], dtype=float)
    tdms_fs = 1.0 / float(ch1.properties["wf_increment"])
    tdms_h = np.sqrt(tx**2 + ty**2)

    run_devices = phone_manifest[
        (phone_manifest["event"] == event)
        & (phone_manifest["nominal_fs"] == nominal_fs)
    ]

    for _, row in run_devices.iterrows():
        _, ts, x, y, _ = load_device(row["path"])
        valid = ts.notna() & x.notna() & y.notna()
        ts = ts[valid].reset_index(drop=True)
        x = x[valid].to_numpy(float)
        y = y[valid].to_numpy(float)
        t = (ts - ts.iloc[0]).dt.total_seconds().to_numpy()
        phone_fs = 1.0 / np.median(np.diff(t))

        xg, yg = x / G0, y / G0
        x_dyn = xg - np.mean(xg)
        y_dyn = yg - np.mean(yg)
        phone_h = np.sqrt(x_dyn**2 + y_dyn**2)
        ref_h = resample_signal(tdms_h, tdms_fs, phone_fs)

        common_stop = common_lag + len(ref_h)
        common_obs = phone_h[common_lag:common_stop]
        common_corr, common_nrmse = fidelity_metrics(ref_h, common_obs)

        optimal_start, _ = best_pearson_start(
            phone_h, ref_h, common_lag, phone_fs, MAX_RESIDUAL_SHIFT_S
        )
        optimal_stop = optimal_start + len(ref_h)
        corrected_corr, corrected_nrmse = fidelity_metrics(
            ref_h, phone_h[optimal_start:optimal_stop]
        )

        residual_samples = optimal_start - common_lag
        residual_s = residual_samples / phone_fs

        pre_n = int(round(NOISE_WINDOW_S * phone_fs))
        pre_end = optimal_start
        pre_start = max(0, pre_end - pre_n)

        if pre_end - pre_start >= max(20, int(phone_fs)):
            noise_rms = dynamic_vector_rms(
                xg[pre_start:pre_end], yg[pre_start:pre_end]
            )
        else:
            noise_rms = np.nan

        event_rms = dynamic_vector_rms(
            xg[optimal_start:optimal_stop],
            yg[optimal_start:optimal_stop],
        )
        snr_db = (
            20 * np.log10(event_rms / noise_rms)
            if np.isfinite(noise_rms) and noise_rms > 0 and event_rms > 0
            else np.nan
        )

        diagnostic_rows.append({
            "event": event,
            "nominal_fs": nominal_fs,
            "device": row["device"],
            "phone_fs": phone_fs,
            "common_lag_samples": common_lag,
            "optimal_device_lag_samples": optimal_start,
            "residual_lag_samples": residual_samples,
            "residual_lag_s": residual_s,
            "abs_residual_lag_s": abs(residual_s),
            "h_corr_common": common_corr,
            "h_corr_residual_corrected": corrected_corr,
            "delta_h_corr": corrected_corr - common_corr,
            "h_nrmse_common": common_nrmse,
            "h_nrmse_residual_corrected": corrected_nrmse,
            "delta_h_nrmse": corrected_nrmse - common_nrmse,
            "pre_event_noise_rms_g": noise_rms,
            "event_dynamic_rms_g": event_rms,
            "snr_db": snr_db,
        })

diagnostics = pd.DataFrame(diagnostic_rows)
if len(diagnostics) != 56:
    raise RuntimeError(f"Expected 56 diagnostic rows, found {len(diagnostics)}")

diagnostic_summary = (
    diagnostics
    .groupby(["device", "nominal_fs"])
    .agg(
        n=("event", "size"),
        median_abs_residual_lag_s=("abs_residual_lag_s", "median"),
        max_abs_residual_lag_s=("abs_residual_lag_s", "max"),
        mean_h_corr_common=("h_corr_common", "mean"),
        mean_h_corr_corrected=("h_corr_residual_corrected", "mean"),
        mean_delta_h_corr=("delta_h_corr", "mean"),
        mean_nrmse_common=("h_nrmse_common", "mean"),
        mean_nrmse_corrected=("h_nrmse_residual_corrected", "mean"),
        median_snr_db=("snr_db", "median"),
        min_snr_db=("snr_db", "min"),
    )
    .reset_index()
)

relationship_rows = []
for setting in ["ALL", 30, 50]:
    subset = diagnostics if setting == "ALL" else diagnostics[diagnostics["nominal_fs"] == setting]
    lag_data = subset[["abs_residual_lag_s", "h_corr_common"]].dropna()
    snr_data = subset[["snr_db", "h_corr_common"]].dropna()

    relationship_rows.append({
        "setting": setting,
        "n": len(subset),
        "pearson_abs_residual_lag_vs_hcorr": (
            np.corrcoef(lag_data["abs_residual_lag_s"], lag_data["h_corr_common"])[0, 1]
            if len(lag_data) >= 3 else np.nan
        ),
        "pearson_snr_vs_hcorr": (
            np.corrcoef(snr_data["snr_db"], snr_data["h_corr_common"])[0, 1]
            if len(snr_data) >= 3 else np.nan
        ),
    })

relationships = pd.DataFrame(relationship_rows)
low_corr = diagnostics.loc[
    diagnostics["h_corr_common"] < 0.50,
    [
        "event", "nominal_fs", "device",
        "h_corr_common", "h_corr_residual_corrected",
        "delta_h_corr", "abs_residual_lag_s", "snr_db",
    ],
].sort_values("h_corr_common")

diagnostics.to_csv(RESULTS_DIR / "ResidualLag_SNR_Recordings.csv", index=False)
diagnostic_summary.to_csv(RESULTS_DIR / "ResidualLag_SNR_Summary.csv", index=False)
relationships.to_csv(RESULTS_DIR / "ResidualLag_SNR_Relationships.csv", index=False)
low_corr.to_csv(RESULTS_DIR / "LowCorrelation_Diagnostics.csv", index=False)

display(diagnostic_summary.round(4))
display(relationships.round(4))
